# 02 - Exploratory Analysis

This notebook aggregates felt reports onto a 1 km grid, builds the modelling
features, and looks at what the data actually shows before any model is fitted.

Four things are examined:

1. How to turn a cell's reported intensities into a target
2. Whether intensity falls with distance the way it should
3. Where the data comes from, and who is missing from it
4. Which features are event-level rather than cell-level, and why that matters

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import clean
import features as feat

pd.set_option("display.width", 120)

In [ ]:
events = pd.read_csv("../data/processed/events.csv")
felt = pd.read_csv("../data/processed/felt_reports.csv")

cells_df = clean.build_cells(felt, min_reports=5, cell_size_m=1000)

## Reporting locations that cannot be used

Two problems show up once the reports are mapped.

The first is that 34 locations sit at exactly longitude 0.005493, latitude
0.002747, one per event, carrying up to 55 reports each. That coordinate is a
few hundred metres from the origin of the coordinate system, in the Gulf of
Guinea. It is what a failed geolocation defaults to. Left in, it would create a
densely observed cell thousands of kilometres from every epicentre.

The second is genuine reports filed from Australia, the United Kingdom, Spain,
the Netherlands and the Philippines. Real people, but not New Zealand shaking.

Together they are only 0.09% of reports, but they are concentrated in ways that
would distort the distance relationship badly.

In [ ]:
category = clean.classify_locations(felt)
summary = (
    felt.assign(category=category)
    .groupby("category")
    .agg(locations=("report_count", "size"), reports=("report_count", "sum"))
)
summary["share_of_reports_%"] = (100 * summary["reports"] / summary["reports"].sum()).round(3)
summary

## Choosing a target

Each cell holds a distribution of reported intensities rather than a single
value, so something has to be chosen. Mean, median and mode are all computed,
and they disagree more than might be expected.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for ax, column, title in zip(axes,
                             ["mmi_mean", "mmi_median", "mmi_mode"],
                             ["Mean", "Median", "Mode"]):
    ax.hist(cells_df[column], bins=40, color="steelblue")
    ax.set_title(f"{title} MMI per cell")
    ax.set_xlabel("MMI")
axes[0].set_ylabel("Cells")
plt.tight_layout()
plt.show()

cells_df[["mmi_mean", "mmi_median", "mmi_mode"]].describe().round(2)

In [ ]:
disagreement = pd.Series({
    "mean vs median differ by 1 or more": (cells_df["mmi_mean"] - cells_df["mmi_median"]).abs().ge(1).mean(),
    "mean vs mode differ by 1 or more": (cells_df["mmi_mean"] - cells_df["mmi_mode"]).abs().ge(1).mean(),
    "median vs mode differ at all": (cells_df["mmi_median"] != cells_df["mmi_mode"]).mean(),
    "median lands on a half step": (cells_df["mmi_median"] % 1 != 0).mean(),
})
(100 * disagreement).round(1).to_frame("percent of cells")

### The problem with the mode

The mode is the natural choice for an ordinal scale, because it returns a real
category rather than an average that nobody reported. But it is unstable, and
it throws away most of the information in a cell.

In [ ]:
counts = cells_df[clean.MMI_COLUMNS].to_numpy()
top_share = counts.max(axis=1) / counts.sum(axis=1)

print(f"cells where the mode is a tie          {100 * cells_df['mode_is_tied'].mean():.1f}%")
print(f"median share of reports backing the mode {100 * np.median(top_share):.1f}%")
print(f"cells where the mode is a minority view  {100 * (top_share < 0.5).mean():.1f}%")

by_size = (
    cells_df.assign(bucket=pd.cut(cells_df["report_count"], [4, 9, 19, 49, 10**9],
                                  labels=["5-9", "10-19", "20-49", "50+"]))
    .groupby("bucket", observed=True)["mode_is_tied"]
    .agg(cells="size", tied_rate="mean")
)
by_size["tied_rate"] = (100 * by_size["tied_rate"]).round(1)
by_size

A tied mode is decided by an arbitrary rule, and ties are concentrated exactly
where the data is thinnest. Raising the minimum reports per cell would reduce
them, but expensively: going from 5 to 20 cuts ties from 12.5% to 4.9% while
discarding three quarters of the cells and 21 of the earthquakes.

### Avoiding the choice

There is a third option, which is not to collapse the cell at all. Every
reported level is kept as its own row, weighted by how many people chose it. A
cell where 12 people said MMI 4 and 9 said MMI 5 contributes both, weighted 12
and 9.

That resolves the whole problem. Every label is a genuine integer report, so
nothing takes an impossible half-step value. There is no tie to break. And
nothing is discarded, which matters given the mode represents a minority of
reports in roughly a third of cells.

In [ ]:
weighted = clean.expand_to_weighted_labels(cells_df)

print(f"cells          {len(cells_df):>8,}")
print(f"weighted rows  {len(weighted):>8,}")
print(f"total weight   {int(weighted['weight'].sum()):>8,}  (equals the retained reports)")
print(f"label values   {sorted(weighted['mmi'].unique())}")

## Does intensity fall with distance?

This is the relationship the whole project rests on. Pooling every report
suggests it barely does.

In [ ]:
cells_df_feat = feat.build_features(
    cells_df, events, vs30_grid=feat.load_vs30_grid("../data/external/vs30_grid.csv"), verbose=False
)
long = clean.expand_to_weighted_labels(
    cells_df_feat, feature_columns=["hypocentral_distance_km", "magnitude", "local_hour", "vs30"]
)

bands = [0, 25, 50, 100, 200, 400, 2000]
long["band"] = pd.cut(long["hypocentral_distance_km"], bands)

pooled = long.groupby("band", observed=True).apply(
    lambda g: np.average(g["mmi"], weights=g["weight"]), include_groups=False
)
pooled.round(2).to_frame("mean reported MMI")

A fall of about one intensity unit across two orders of magnitude of distance
is far too shallow to be physical. Something is masking the relationship.

The cause is that magnitude and distance are entangled. Only large earthquakes
are felt far away, so the far field is populated almost entirely by big events
that shake hard, which flattens the pooled curve.

In [ ]:
far = long[long["hypocentral_distance_km"] > 320]
near = long[long["hypocentral_distance_km"] <= 100]

print(f"share of reports beyond 320 km coming from M6+  {100 * far.loc[far['magnitude'] >= 6, 'weight'].sum() / far['weight'].sum():.1f}%")
print(f"share of reports within 100 km coming from M6+  {100 * near.loc[near['magnitude'] >= 6, 'weight'].sum() / near['weight'].sum():.1f}%")

In [ ]:
long["mag_band"] = pd.cut(long["magnitude"], [4, 5, 6, 7, 8],
                          labels=["M4-5", "M5-6", "M6-7", "M7+"])

stratified = long.groupby(["mag_band", "band"], observed=True).apply(
    lambda g: np.average(g["mmi"], weights=g["weight"]) if g["weight"].sum() > 200 else np.nan,
    include_groups=False,
).unstack()

fig, ax = plt.subplots(figsize=(8, 4.5))
for label, row in stratified.iterrows():
    ax.plot(range(len(row)), row.values, marker="o", label=label)
ax.set_xticks(range(len(stratified.columns)))
ax.set_xticklabels([str(c) for c in stratified.columns], rotation=45)
ax.set_xlabel("Hypocentral distance (km)")
ax.set_ylabel("Mean reported MMI")
ax.set_title("Intensity falls with distance once magnitude is held roughly constant")
ax.legend()
plt.tight_layout()
plt.show()

stratified.round(2)

Held within a magnitude band, the decay appears and is monotonic for M5 and
above. The pooled view was a confound, not a finding.

The M4 to M5 line is different: intensity appears to *rise* with distance,
which is impossible. Two events dominate those far-field cells, and both are
part of the November 2016 Kaikoura aftershock sequence. During a sequence
people cannot tell which shake they are reporting, so reports get attributed to
the wrong event. It is a reminder that the labels carry the reporter's
interpretation, not just the ground motion.

In [ ]:
odd = cells_df_feat[(cells_df_feat["magnitude"] < 5) & (cells_df_feat["hypocentral_distance_km"] > 200)]
(odd.groupby("public_id")
    .agg(magnitude=("magnitude", "first"), depth_km=("depth_km", "first"),
         cells=("cell_x", "size"), furthest_km=("hypocentral_distance_km", "max"))
    .sort_values("cells", ascending=False)
    .head()
    .round(1))

## Who is missing from the data

Felt reports come from people, so the dataset records where people are as much
as where the ground shook. Two measurements make that concrete.

In [ ]:
grid = feat.load_vs30_grid("../data/external/vs30_grid.csv")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(grid["vs30"], bins=40, alpha=0.6, density=True, label="National grid")
ax.hist(cells_df_feat["vs30"].dropna(), bins=40, alpha=0.6, density=True, label="Reporting cells")
ax.set_xlabel("Vs30 (m/s), lower means softer ground")
ax.set_ylabel("Density")
ax.set_title("Reports come from soft ground, because that is where towns are")
ax.legend()
plt.tight_layout()
plt.show()

print(f"mean Vs30 nationally      {grid['vs30'].mean():.0f} m/s")
print(f"mean Vs30 under cells     {cells_df_feat['vs30'].mean():.0f} m/s")
print(f"cells cover {cells_df_feat.groupby(['cell_x', 'cell_y']).ngroups:,} distinct square kilometres")

Reporting cells sit on ground averaging 186 m/s softer than the country as a
whole, because settlements are built on sedimentary basins. Softer ground
amplifies shaking, so Vs30 and population density are confounded from the
outset. Any Vs30 coefficient the model learns will carry both effects, and
attributing it purely to site response would be wrong.

The coverage figure is the blunter point: a few thousand square kilometres out
of roughly 268,000. The data describes shaking where people live, and is silent
almost everywhere else.

There is a subtler version of the same problem. A felt report is only filed by
someone who felt something. Nobody submits a report to say they felt nothing,
so the absence of reports cannot be read as absence of shaking, and the reports
that do arrive from distant places are the upper tail of what was experienced
there.

## Which features actually vary

A structural point that matters for how the model is validated. Most features
are properties of the earthquake, not of the cell, so they take one value
across every cell of an event.

In [ ]:
variation = pd.DataFrame({
    "distinct values overall": [cells_df_feat[c].nunique() for c in
                                ["magnitude", "depth_km", "local_hour", "log_hypocentral_distance",
                                 "azimuth_degrees", "vs30"]],
    "varies within an event": [
        bool(cells_df_feat.groupby("public_id")[c].nunique().gt(1).any()) for c in
        ["magnitude", "depth_km", "local_hour", "log_hypocentral_distance",
         "azimuth_degrees", "vs30"]],
}, index=["magnitude", "depth_km", "local_hour", "log_hypocentral_distance",
          "azimuth_degrees", "vs30"])
variation

Magnitude, depth and time of day are constant within an earthquake. With 95
events, those features carry 95 independent observations between them, not
24,241.

That has two consequences. Any split that puts cells from the same earthquake
in both training and test sets lets the model recognise events rather than
learn physics, so the split in the next notebook has to be made by earthquake.
And time of day in particular is close to an event identifier here, so it needs
watching: if it earns a large coefficient, that is more likely memorisation
than a discovery about human reporting behaviour.

## Summary

- The mode is unstable and discards most of a cell's information. Weighting
  every reported level avoids choosing a summary at all, and keeps the target
  on genuine integer values.
- Intensity does fall with distance, but only once magnitude is controlled for.
  Pooled across events the relationship almost disappears.
- Reports come from soft, populated ground, so Vs30 and population are
  confounded before any model is fitted.
- Most features are event-level, so the effective sample size is far smaller
  than the row count suggests, and validation must split by earthquake.

In [ ]:
cells_df_feat.to_csv("../data/processed/features.csv", index=False)
print(f"{len(cells_df_feat):,} rows written to data/processed/features.csv")